# 📓 **Notebook 3 — Phases 4 & 5**

**Project:** [Medical Transcription Classfier and Semantic Search Engine Project](https://github.com/bahadirkoko/medical-transcription-classifier)

## Medical Transcription Analysis — Phases 4 & 5

This notebook builds two things on top of the cleaned dataset from Phase 1-2:

1. **Phase 4 — Semantic Search Engine:** embed clinical notes as dense vectors,
   store them in a vector database (ChromaDB), and classify new queries by
   retrieving the most semantically similar chunks and voting on their specialty.

2. **Phase 5 — Retrieval Validation:** evaluate the search engine against
   held-out ground-truth labels using Precision@K and a retrieval confusion matrix.

**Key distinction from Phase 3:** the Phase 3 classifier *trains* on labeled data
and learns parameters. This system *trains nothing* — it stores the data and
references it at query time (non-parametric / "lazy" learning). It also handles
synonyms that TF-IDF cannot: "kidneys not filtering" → Nephrology, even though
the word "renal" never appears in the query.

### 🗂️ **Project notebooks**
1. Notebook 1 — Phases 1 & 2: Collection & Cleaning, Preprocessing for NLP, TF-IDF, Chi 2
2. Notebook 2 — Phase 3: Model Trainings and Evaluation 
3. **Notebook 3** — Phases 4 & 5: Semantic Search Engine, Retrieval Validation **(you are here)**

## Imports & Data Load

In [ ]:
# data manipulation
import pandas as pd
import numpy as np
import os                               # file path checks (cache guard for embeddings)

# visualization
import matplotlib.pyplot as plt

# text utilities
from collections import Counter         # majority vote in mini-RAG (count specialty occurrences)

# machine learning — validation metrics
from sklearn.model_selection import train_test_split    # held-out split (index vs query set)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # retrieval confusion matrix

# semantic search — the two new libraries for Phase 4
from sentence_transformers import SentenceTransformer   # converts text to 384-D vectors
import chromadb                         # vector database — stores and searches embeddings

# load the processed dataset from Phase 1
df = pd.read_csv('../data/processed/cleaned_medical_notes.csv')
print("shape:", df.shape)
print("columns:", df.columns.tolist())



## Data Filtering

We apply the same specialty filtering as Phase 3 — removing document-format
categories (not genuine medical specialties), Surgery (95% cross-listed with
other specialties — a procedural label, not a distinct clinical vocabulary),
and Radiology (95% of its notes also appear under a real clinical specialty,
confirming it is a service label rather than a distinct classification target).

This gives us a clean set of genuine clinical specialties for the search engine.

In [ ]:
# document-format categories and structurally-redundant specialty labels
not_specialties = [
    'SOAP / Chart / Progress Notes', 'Office Notes', 'Letters',
    'Discharge Summary', 'Emergency Room Reports',
    'Consult - History and Phy.', 'IME-QME-Work Comp etc.',
    'Surgery',    # 95% cross-listed with real specialties
    'Radiology'   # 95% cross-listed — service label, not a clinical specialty
]

df = df[~df['medical_specialty'].isin(not_specialties)].reset_index(drop=True) #mask specialities that are not real specialities.
print("rows after filtering:", len(df))
print("specialties remaining:", df['medical_specialty'].nunique())

---
## Phase 4 — Vector DBs & Semantic Retrieval

### 4.1 The Conceptual "Why"

Traditional TF-IDF models (Phase 3) treat every word as an independent token.
"Renal" and "Kidney" are two completely unrelated columns in the matrix — the
model has no idea they mean the same thing. So a query saying "renal failure"
and a note saying "kidney disease" share zero overlap and won't match.

**Embeddings** solve this by representing text as coordinates in a
high-dimensional space (384 dimensions for our model), trained so that
semantically similar text lands *near each other* in that space. "Renal" and
"Kidney" end up as nearby points; "Renal" and "Banana" end up far apart.
Distance = dissimilarity of meaning. This is what allows synonym-matching.

**The pipeline:**
1. Split notes into chunks (manageable pieces for the embedding model).
2. Embed each chunk → a 384-dimensional vector.
3. Store vectors + metadata in ChromaDB (the vector database).
4. At query time: embed the query, find the nearest stored vectors, vote on
   their specialty labels → prediction.

### 4.2 Step 0 — Held-Out Split (Validation Foundation)

Before building the index, we split the data: 95% goes into the vector store
(the "index set"), and 5% is held out as unseen query notes for Phase 5
validation. This is critical — if we indexed all notes and then queried with
the same notes, the system would trivially retrieve copies of itself (perfect
self-match), making the validation meaningless. The held-out notes were never
indexed, so retrieval performance is genuine.

In [ ]:
# split BEFORE indexing — held-out notes must never enter the vector store
# if we indexed all notes and queried with the same ones, the system would
# trivially retrieve copies of itself (perfect self-match), making validation meaningless

index_df, holdout_df = train_test_split(
    df,
    test_size=0.05,                          # 5% held out (~140 notes) as unseen queries
    stratify=df['medical_specialty'],         # keep specialty proportions equal in both sets, preserve class proportions
    random_state=42                           # fixed seed — same split every run
)

index_df   = index_df.reset_index(drop=True)    # clean 0,1,2... index after split
holdout_df = holdout_df.reset_index(drop=True)  # same for holdout

print("index notes (go into vector store):", len(index_df))
print("holdout notes (used for validation):", len(holdout_df))
print("\nspecialties in holdout set:")
print(holdout_df['medical_specialty'].value_counts())

### 4.2 Step 1 — Chunking Strategy

Embedding models have a limited context window — feeding a 2,000-word note
in whole causes the model to "lose" early context by the end, producing a
blurry, averaged-out vector. We split each note into **200-word chunks** with
a **20-word overlap** between consecutive chunks.

The overlap prevents context from being severed mid-sentence at chunk
boundaries — like overlapping floorboards so there's no gap at the seam.

Each chunk carries its original `medical_specialty` and `note_id` as metadata —
crucial for the voting step in the mini-RAG classifier.

In [ ]:
def chunk_text(text, chunk_size=200, overlap=20):
    """
    Split a long text into overlapping word chunks.
    
    Why chunk, Embedding models have a limited context window — feeding a 
    2000-word note whole causes early context to be "forgotten" by the end,
    producing a blurry averaged vector. Smaller chunks = focused, accurate vectors.
    
    Why overlap? Without it, a sentence spanning the boundary between chunk 1
    and chunk 2 gets cut in half — neither chunk has the full thought. 
    The 20-word overlap ensures context is never severed mid-sentence.
    """
    words = text.split()                           # split on whitespace into a word list
    chunks = []
    step = chunk_size - overlap                    # advance 180 words each iteration (not 200)
                                                   # so consecutive chunks share 20 words
    for start in range(0, len(words), step):       # start positions: 0, 180, 360, 540...
        chunk = words[start : start + chunk_size]  # grab up to 200 words from this position
        if len(chunk) == 0:                        # safety guard for empty trailing slice
            break
        chunks.append(' '.join(chunk))             # rejoin word list back into a string
    return chunks                                  # list of chunk strings, each ≤200 words

# verify the function works on one real note
sample = df['transcription'].iloc[0]               # grab the first note
pieces = chunk_text(sample)
print(f"sample note: {len(sample.split())} words → {len(pieces)} chunks")
# expected: a ~400-word note → ~3 chunks (400/180 ≈ 2.2, rounds up)
# a short note (<200 words) → 1 chunk (the whole note fits in one slice)

In [ ]:
# apply chunking to ALL index notes, attaching metadata to each chunk
# three parallel lists — index i in all three lists describes the same chunk:
# chunk_texts[i] came from note chunk_note_ids[i] with label chunk_specialties[i]
chunk_texts = []        # the actual text of each chunk (what gets embedded)
chunk_specialties = []  # the specialty label carried from the parent note
chunk_note_ids = []     # the parent note's row index (for tracing back to the original)

for note_id, row in index_df.iterrows():   # walk every note: note_id = row index, row = data
    pieces = chunk_text(row['transcription'])   # split this note into ≤200-word chunks
    for piece in pieces:                        # for each chunk produced by this note...
        chunk_texts.append(piece)                       # ...store the chunk text
        chunk_specialties.append(row['medical_specialty'])  # ...carry the specialty label
        chunk_note_ids.append(note_id)                  # ...carry the note's id

print(f"total notes indexed: {len(index_df)}")
print(f"total chunks created: {len(chunk_texts)}")
print(f"average chunks per note: {len(chunk_texts)/len(index_df):.1f}")
# if average is ~5-8, that's expected for ~450-word notes split into 200-word chunks

### 4.2 Step 2 — Generating Dense Vectors

We use `all-MiniLM-L6-v2` from SentenceTransformers — a lightweight model that
produces **384-dimensional vectors** balancing speed and quality. Each chunk is
transformed into a point in 384-D meaning-space.

The cache guard below avoids recomputing embeddings on every kernel restart —
embedding ~10,000+ chunks takes a few minutes on CPU. Cached embeddings load
in milliseconds.

**Important:** embeddings are computed on the **raw transcription text** (not
`clean_text`). Embedding models are trained on natural language and perform
better with full sentences, stopwords, and punctuation intact.

In [ ]:
emb_path = '../data/processed/embeddings.npy'

if os.path.exists(emb_path):
    # load cached embeddings — instant on subsequent runs
    embeddings = np.load(emb_path)
    print("loaded cached embeddings:", embeddings.shape)
    # safety check: embeddings must match current chunks
    assert embeddings.shape[0] == len(chunk_texts), \
        f"stale cache! embeddings={embeddings.shape[0]}, chunks={len(chunk_texts)} — delete embeddings.npy and recompute"
else:
    # first run — compute and cache (takes a few minutes on CPU)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(chunk_texts, show_progress_bar=True, batch_size=64)
    np.save(emb_path, embeddings)
    print("computed and saved:", embeddings.shape)

# shape should be (num_chunks, 384)
print(f"embedding matrix: {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions")

### 4.2 Step 3 — Building the Vector Store (ChromaDB)

ChromaDB is a vector database that stores our chunk vectors alongside their
metadata (specialty, note_id). It provides fast nearest-neighbor search using
**cosine similarity** — measuring the angle between two vectors (are they
pointing the same direction in meaning-space?).

We use `PersistentClient` so the database survives kernel restarts — built once,
reconnected instantly on subsequent runs. The batched `add` handles ChromaDB's
per-call size limit of ~5,000 items.

In [ ]:
# PersistentClient writes the DB to a folder on disk — survives kernel restarts
# unlike chromadb.Client() which is in-memory only and resets every session
client = chromadb.PersistentClient(path='../data/processed/chroma_db')

# list all collections currently saved in this DB
# a "collection" is a named container of vectors + metadata (like a table in a database)
existing = [c.name for c in client.list_collections()]

if "medical_notes" in existing:
    # collection already built and saved from a previous run — just reconnect
    # this path is instant: no re-embedding, no re-adding, just opens the saved DB
    collection = client.get_collection("medical_notes")
    print("reconnected to existing DB, count:", collection.count())

else:
    # first run only — build the collection from scratch
    collection = client.create_collection("medical_notes")

    # convert numpy array to plain Python list — ChromaDB requires standard Python types
    all_embeddings = embeddings.tolist()

    # metadata: one dict per chunk, carrying specialty + note_id
    # str() and int() ensure ChromaDB-compatible types (no numpy int64 / pandas strings)
    all_metadatas = [{"specialty": str(s), "note_id": int(n)}
                     for s, n in zip(chunk_specialties, chunk_note_ids)]

    # ChromaDB requires a unique string ID per chunk — we use the position index
    all_ids = [str(i) for i in range(len(chunk_texts))]

    # ChromaDB's add() has a per-call size limit (~5461 items)
    # we batch in groups of 5000 to stay safely under that limit
    BATCH = 5000
    for start in range(0, len(chunk_texts), BATCH):
        end = start + BATCH
        collection.add(
            embeddings=all_embeddings[start:end],   # the vectors
            documents=chunk_texts[start:end],        # the raw text (for inspection)
            metadatas=all_metadatas[start:end],      # specialty + note_id per chunk
            ids=all_ids[start:end]                   # unique string id per chunk
        )
        print(f"added chunks {start} to {min(end, len(chunk_texts))}")

    print("built new DB, count:", collection.count())
    # count should match len(chunk_texts) — confirms all chunks were stored

---
### 4.3 — Mini-RAG Classifier (Non-Parametric)

This is the semantic classifier. Unlike Phase 3's trained models, it **learns
nothing** — it stores the data and references it at query time:

1. **Embed** the query using the same MiniLM model (same 384-D space as the stored chunks).
2. **Search** ChromaDB for the top-5 most similar chunks by cosine similarity.
3. **Vote** — the majority specialty among those 5 chunks is the prediction.

This is called "non-parametric" because there are no learned parameters — the
data *is* the model. Remove the vector store and the classifier has nothing.
Contrast with Phase 3 where the trained model can predict without the original data.

In [ ]:
def classify_by_retrieval(query, k=5):
    """
    Non-parametric specialty classifier using semantic retrieval.
    
    No training involved — finds the k most semantically similar chunks
    in the vector store and votes on their specialty labels.
    
    Args:
        query: plain-text clinical description (any phrasing, even layperson)
        k:     number of nearest chunks to retrieve and vote on (default 5)
    
    Returns:
        vote:                 the predicted specialty (majority of k retrieved)
        retrieved_specialties: the full list of k specialty labels (for inspection)
    """

    # step 1 — embed the query into the same 384-D space as the stored chunks
    # using a DIFFERENT model would produce incompatible coordinates — no valid comparison
    # model.encode() expects a list, so we wrap the single query string in []
    # .tolist() converts numpy array → plain Python list (ChromaDB requirement)
    query_vector = model.encode([query]).tolist()

    # step 2 — search the vector store for the k most similar chunks
    # ChromaDB compares query_vector against all stored vectors using cosine similarity
    # (cosine similarity measures the angle between vectors — 1.0 = identical direction)
    # returns the k chunks whose vectors point most similarly to the query vector
    results = collection.query(
        query_embeddings=query_vector,  # the query's 384-D coordinates
        n_results=k                     # how many nearest neighbors to return
    )

    # step 3 — extract the specialty label from each retrieved chunk's metadata
    # results['metadatas'] is wrapped in an extra list (ChromaDB supports multi-query)
    # [0] unwraps the single-query layer → gives a list of k metadata dicts
    # each dict has 'specialty' and 'note_id' (what we stored during indexing)
    retrieved_specialties = [m['specialty'] for m in results['metadatas'][0]]

    # step 4 — majority vote: which specialty appears most among the k results?
    # Counter({'Neurology': 3, 'Radiology': 1, 'Surgery': 1})
    # .most_common(1) → [('Neurology', 3)]   (list of (value, count) tuples)
    # [0][0]          → 'Neurology'           (unwrap to just the specialty string)
    vote = Counter(retrieved_specialties).most_common(1)[0][0]

    return vote, retrieved_specialties  # prediction + full vote breakdown

### Demo — Semantic Query Testing

Two sets of queries demonstrate the system's capabilities:

**Standard queries** — medical phrasing to confirm basic retrieval works.

**Synonym / layperson queries** — the key demonstration. These use everyday
language with *zero* exact medical keywords. If the system correctly routes
"the bone in my lower leg snapped" → Orthopedic, it proves semantic
understanding beyond keyword matching. This is something TF-IDF (Phase 3)
*cannot* do — it would see zero word overlap with stored notes.

In [ ]:
# standard medical queries — confirm basic retrieval
test_queries = [
    "Patient has persistent cough and wheezing",
    "the patient's kidneys are not filtering properly",   # synonym test: no "renal"
    "severe chest pain radiating to the left arm",
    "fracture of the femur after a fall",
    "removal of the gallbladder due to stones",
    "child presents for routine vaccination and growth check",
    "patient reports feeling depressed and hopeless for weeks",
    "blurry vision and cataract in the right eye",
]

for q in test_queries:
    pred, votes = classify_by_retrieval(q, k=5)
    print(f"\nQUERY:     {q}")
    print(f"VOTES:     {votes}")
    print(f"PREDICTED: {pred}")

In [ ]:
# layperson phrasing — zero clinical keywords
# this is the headline demonstration: semantic search handles synonyms
# that keyword/TF-IDF search would completely miss
print("=== LAYPERSON SYNONYM TEST ===\n")
layperson_queries = [
    "my heart skips beats and races sometimes",         # no "arrhythmia" or "cardiac"
    "trouble breathing and a whistling sound when I exhale",  # no "asthma" or "wheeze"
    "the bone in my lower leg snapped",                 # no "fracture" or "tibia"
]
for q in layperson_queries:
    pred, votes = classify_by_retrieval(q)
    print(f"'{q}'")
    print(f"  → predicted: {pred}  |  votes: {votes}\n")

---
## Phase 5 — Retrieval Validation: Ground Truth Test

We now systematically evaluate the search engine against real labels using the
held-out notes (never indexed — see split in Step 0).

**Why held-out notes?** If we queried with notes that *are* in the index, the
system would retrieve the note itself (perfect self-match) — inflating scores
trivially. Held-out notes give honest, genuine retrieval performance.

**Two metrics:**
- **Precision@5:** for each query, what fraction of the 5 retrieved chunks
  match the true specialty? Averaged across all held-out notes.
- **Retrieval Confusion Matrix:** query's true label vs. majority-vote of
  retrieved chunks — reveals semantic overlaps between specialties.

### 5.2 — Precision@K

In [ ]:
def precision_at_k(query_text, true_specialty, k=5):
    """
    Compute Precision@K for one query note.
    
    Asks: "of the k chunks retrieved for this query, what fraction
    carry the correct specialty label?"
    
    Example: true_specialty = "Neurology"
             retrieved      = [Neurology, Radiology, Neurology, Neurology, Surgery]
             matches        = 3  →  Precision@5 = 3/5 = 0.60 (60%)
    
    Args:
        query_text:     the transcription text used as the search query
        true_specialty: the ground-truth specialty label for this note
        k:              number of chunks to retrieve (default 5)
    
    Returns:
        matches/k:  precision score (0.0 to 1.0)
        retrieved:  the list of k retrieved specialty labels (for inspection)
    """

    # embed the query — same model, same 384-D space as stored chunks
    qvec = model.encode([query_text]).tolist()

    # retrieve the k most similar chunks from the vector store
    results = collection.query(query_embeddings=qvec, n_results=k)

    # extract specialty labels from the k retrieved chunks' metadata
    retrieved = [m['specialty'] for m in results['metadatas'][0]]

    # count how many retrieved specialties exactly match the true label
    # sum(1 for ...) is a generator expression — adds 1 for each True condition
    # equivalent to: len([s for s in retrieved if s == true_specialty])
    matches = sum(1 for s in retrieved if s == true_specialty)

    return matches / k, retrieved   # precision score (0.0–1.0) + full retrieved list


# ── evaluate over ALL held-out notes ────────────────────────────────────────

precisions = []   # collect one precision score per held-out note

for _, row in holdout_df.iterrows():          # walk every held-out query note
    p, _ = precision_at_k(
        row['transcription'],                  # the note text is the query
        row['medical_specialty']               # the label is the ground truth
    )
    precisions.append(p)                       # collect the score

# ── report results ───────────────────────────────────────────────────────────

print(f"Mean Precision@5: {np.mean(precisions):.2%}")
# average across all held-out queries — the headline validation metric

print(f"Best:  {max(precisions):.0%}  |  Worst: {min(precisions):.0%}")
# 100% = all 5 retrieved chunks matched the true specialty (distinctive vocabulary)
# 0%   = none of the 5 matched (rare/overlapping specialty, or cross-listed note)

print(f"\nContext: random baseline for {df['medical_specialty'].nunique()} classes ≈ "
      f"{1/df['medical_specialty'].nunique()*100:.1f}%")
# random chance: 1 / number_of_classes — our score should be well above this
# e.g. 26 classes → 3.8% random baseline; 64% result = ~16× better than chance

**Interpreting Precision@5:** a random baseline across ~26 specialties would
be ~3.8%. Our system at ~64% is roughly 16× better than chance, confirming
semantic retrieval is finding genuinely relevant notes. The 0–100% spread
reflects the dataset's class imbalance — well-represented specialties with
distinctive vocabulary score near 100%; rare or semantically-overlapping
specialties score lower.

This metric scores at the *chunk* level (retrieved chunks, not unique notes),
and uses exact-label matching. Off-diagonal retrievals are often
*medically reasonable neighbors* (e.g. Neurology↔Neurosurgery both discuss
spinal pathology), so Precision@5 understates true retrieval quality.

### 5.3 — Retrieval Confusion Matrix

Same visualization as Phase 3 but for the search engine — query's true label
vs. majority-vote of retrieved chunks. Off-diagonal cells reveal **semantic
overlaps**: pairs of specialties whose clinical language is genuinely similar.
These are not failures; they reflect real multi-disciplinary medicine.

In [ ]:
true_labels, voted_labels = [], []

for _, row in holdout_df.iterrows():
    # retrieve and vote for each held-out note
    _, retrieved = precision_at_k(row['transcription'], row['medical_specialty'])
    vote = Counter(retrieved).most_common(1)[0][0]
    true_labels.append(row['medical_specialty'])
    voted_labels.append(vote)

# build normalized confusion matrix (rows sum to 1.0 — proportions per true class)
labels = sorted(set(true_labels) | set(voted_labels))
cm = confusion_matrix(true_labels, voted_labels, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(16, 14))
ConfusionMatrixDisplay(
    cm.round(2),
    display_labels=labels
).plot(
    ax=ax,
    xticks_rotation=90,
    cmap='Blues',
    values_format='.2f'     # 2 decimal places per cell
)
plt.title('Retrieval Confusion Matrix — Semantic Search (Phase 5)')
plt.tight_layout()
plt.savefig('../outputs/figures/retrieval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

**Reading the matrix:** bright diagonal cells = correct specialty retrieval.
Off-diagonal hotspots = semantic overlaps — pairs where the embedding model
perceives genuine linguistic similarity. Compare these pairs to the Phase 3
classifier confusion matrix: if the same pairs confuse both a *trained*
classifier and an *untrained* semantic retriever, that is strong evidence the
overlap is **inherent to the data** (those specialties genuinely share
clinical vocabulary), not an artifact of either method.

### 5.4 — Holdout Coverage Check

In [ ]:
# check how many specialties are represented in the holdout query set
print("Specialties in query (holdout) set:")
print(holdout_df['medical_specialty'].value_counts())
print(f"\nQuery set covers {holdout_df['medical_specialty'].nunique()} of "
      f"{df['medical_specialty'].nunique()} specialties")
print(f"\nNote: at 5% holdout with heavy imbalance, rare specialties have "
      f"very few query notes — Precision@5 is most reliable for well-represented classes.")

---
## Summary

| Component | Method | Key Finding |
|---|---|---|
| Embedding model | all-MiniLM-L6-v2 (384-D) | Synonym-aware: "kidneys not filtering" → Nephrology |
| Vector store | ChromaDB (cosine similarity) | 384-D meaning-space, persistent on disk |
| Classifier | Mini-RAG (retrieve + vote) | No training — references data at query time |
| Precision@5 | ~34% (vs ~3.8% random) | ~9× better than chance across 26 specialties |
| Key insight | Semantic overlaps = medical reality | Same pairs confuse Phase 3 AND Phase 5 — inherent in data |

**Limitation:** Precision@5 uses exact-label matching — medically-reasonable
neighbors (Neurology retrieving Neurosurgery on a spinal case) are counted as
misses. True clinical utility is higher than the number suggests.